# LOADING DATA

In [4]:
import pandas as pd
import pickle
from pathlib import Path

# Get the Code directory (project root)
current_dir = Path.cwd()  # from_scratch directory
code_dir = current_dir.parent.parent.parent.parent  # Go up to Code directory
print(f"Code directory: {code_dir}")

# Define all data paths directly
PATHS = {
    # Training features
    'X_train_smote': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_smote.parquet',
    'X_train_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_tomek.parquet',
    'X_train_smote_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_smote_tomek.parquet',
    
    # Training targets
    'y_train_smote': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'smote' / 'y_smote.pkl',
    'y_train_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'tomek' / 'y_tomek.pkl',
    'y_train_smote_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'smote_tomek' / 'y_smote_tomek.pkl',
    
    # Validation and test features
    'X_val': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'val' / 'X_val.parquet',
    'X_test': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'test' / 'X_test.parquet',
    
    # Validation and test targets
    'y_val': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'val' / 'y_val.pkl',
    'y_test': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'test' / 'y_test.pkl',
}

# Print all paths for verification
print("\nData paths:")
for key, path in PATHS.items():
    exists = "✓" if path.exists() else "✗"
    print(f"  {exists} {key}: {path}")

# Load all data
def load_all_data():
    """Load all data files"""
    data = {}
    
    print("\n" + "="*50)
    print("LOADING DATA")
    print("="*50)
    
    # Load parquet files
    parquet_keys = ['X_train_smote', 'X_train_tomek', 'X_train_smote_tomek', 'X_val', 'X_test']
    for key in parquet_keys:
        path = PATHS[key]
        if path.exists():
            try:
                data[key] = pd.read_parquet(path)
                print(f"✓ Loaded {key}: {data[key].shape}")
            except Exception as e:
                print(f"✗ Error loading {key}: {e}")
        else:
            print(f"✗ Skipping {key}: file not found at {path}")
    
    # Load pickle files
    pickle_keys = ['y_train_smote', 'y_train_tomek', 'y_train_smote_tomek', 'y_val', 'y_test']
    for key in pickle_keys:
        path = PATHS[key]
        if path.exists():
            try:
                with open(path, 'rb') as f:
                    data[key] = pickle.load(f)
                print(f"✓ Loaded {key}: {len(data[key])} samples")
            except Exception as e:
                print(f"✗ Error loading {key}: {e}")
        else:
            print(f"✗ Skipping {key}: file not found at {path}")
    
    return data

# Load the data
data = load_all_data()

if data:
    print(f"\n" + "="*50)
    print(f"Successfully loaded {len(data)} datasets")
    print("="*50)
    for key, value in data.items():
        if hasattr(value, 'shape'):
            print(f"  {key}: {value.shape}")
        else:
            print(f"  {key}: {len(value)} samples")
else:
    print("\nNo data was loaded")

Code directory: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code

Data paths:
  ✓ X_train_smote: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Normalized_Data_split\X_train_smote.parquet
  ✓ X_train_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Normalized_Data_split\X_train_tomek.parquet
  ✓ X_train_smote_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Normalized_Data_split\X_train_smote_tomek.parquet
  ✓ y_train_smote: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Resampled_Data_split\smote\y_smote.pkl
  ✓ y_train_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Resampled_Data_split\tomek\y_tomek.pkl
  ✓ y_train_smote_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessi

# DT

In [5]:
import numpy as np
import pandas as pd
import time
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Use the existing calculate_accuracy function
def calculate_accuracy(y_true, y_pred):
    """Calculate the accuracy score using sklearn."""
    if isinstance(y_true, pd.Series):
        y_true = y_true.values
    return accuracy_score(y_true, y_pred)

# --- Training and Evaluation Function ---

def train_and_evaluate_dt(data, train_key_X, train_key_y, val_key_X, val_key_y, model_params):
    """Utility function to train and evaluate the Scikit-learn Decision Tree on a specific dataset."""
    dataset_name = train_key_X.upper().replace('X_TRAIN_', '')
    print(f"\n--- Training Decision Tree on **{dataset_name}** Dataset ---")
    
    # Prepare data
    X_train = data[train_key_X]
    y_train = data[train_key_y]
    X_val = data[val_key_X]
    y_val = data[val_key_y]
    
    # Initialize and train the model
    dt = DecisionTreeClassifier(**model_params)
    start_time = time.time()
    
    # Fit the model
    dt.fit(X_train, y_train)
    
    end_time = time.time()
    
    print(f"\nTraining completed in **{end_time - start_time:.2f} seconds**.")
    
    # Make predictions
    y_train_pred = dt.predict(X_train)
    y_val_pred = dt.predict(X_val)
    
    # Evaluate
    train_acc = calculate_accuracy(y_train, y_train_pred)
    val_acc = calculate_accuracy(y_val, y_val_pred)
    
    # Decision Trees often result in perfect training accuracy if not pruned (max_depth=None)
    print(f"**Training Accuracy:** {train_acc:.4f}") 
    print(f"**Validation Accuracy:** {val_acc:.4f}")

    # Optional: Print detailed report for validation set
    print("\nClassification Report (Validation):")
    print(classification_report(y_val, y_val_pred))
    
    return dt, train_acc, val_acc

# Define model parameters
# We limit the depth to prevent extreme overfitting, which is common with single trees.
DT_PARAMS = {
    'max_depth': 10,                 # Limit the depth of the tree
    'criterion': 'gini',             # Use Gini impurity for splitting
    'min_samples_leaf': 5,           # Minimum number of samples required to be at a leaf node
    'random_state': 42
}

# List of datasets to process
datasets = [
    ('X_train_smote', 'y_train_smote'),
    ('X_train_tomek', 'y_train_tomek'),
    ('X_train_smote_tomek', 'y_train_smote_tomek'),
]

results = {}
val_features_key = 'X_val'
val_target_key = 'y_val'

# Run the training loop for all sampled datasets
for X_key, y_key in datasets:
    model, train_acc, val_acc = train_and_evaluate_dt(
        data, 
        X_key, 
        y_key, 
        val_features_key, 
        val_target_key, 
        DT_PARAMS
    )
    # Store results
    dataset_name = X_key.replace('X_train_', '')
    results[dataset_name] = {
        'model': model,
        'train_accuracy': train_acc,
        'val_accuracy': val_acc
    }

# --- Final Summary ---
print("\n" + "="*70)
print("FINAL SCIKIT-LEARN DECISION TREE TRAINING RESULTS")
print("="*70)
for name, res in results.items():
    print(f"🌳 **{name.upper()}**:")
    print(f"  - Training Accuracy: {res['train_accuracy']:.4f}")
    print(f"  - Validation Accuracy: {res['val_accuracy']:.4f}")
    print("-" * 25)


--- Training Decision Tree on **SMOTE** Dataset ---

Training completed in **2.25 seconds**.
**Training Accuracy:** 0.9374
**Validation Accuracy:** 0.8749

Classification Report (Validation):
              precision    recall  f1-score   support

           0       0.98      0.89      0.93      3276
           1       0.27      0.69      0.39       201

    accuracy                           0.87      3477
   macro avg       0.62      0.79      0.66      3477
weighted avg       0.94      0.87      0.90      3477


--- Training Decision Tree on **TOMEK** Dataset ---

Training completed in **0.59 seconds**.
**Training Accuracy:** 0.9681
**Validation Accuracy:** 0.9382

Classification Report (Validation):
              precision    recall  f1-score   support

           0       0.96      0.98      0.97      3276
           1       0.45      0.33      0.38       201

    accuracy                           0.94      3477
   macro avg       0.71      0.65      0.68      3477
weighted avg   

# Bayesian Optimisation

In [6]:
!pip install bayesian-optimization

In [7]:
import numpy as np
import pandas as pd
import time
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from bayes_opt import BayesianOptimization
from sklearn.model_selection import cross_val_score

# --- Define the Objective Function for BO ---

def dt_cv_score(max_depth, min_samples_split, min_samples_leaf):
    """
    Objective function for Bayesian Optimization for the Decision Tree.
    It returns the cross-validation score based on suggested hyperparameters.
    """
    
    # 1. Cast parameters to correct types (BO suggests floats)
    max_depth = int(round(max_depth))
    min_samples_split = int(round(min_samples_split))
    min_samples_leaf = int(round(min_samples_leaf))

    # Ensure constraints are met (split >= leaf)
    min_samples_split = max(2, min_samples_split) # Split must be at least 2
    min_samples_leaf = max(1, min_samples_leaf)   # Leaf must be at least 1
    
    # 2. Define the Decision Tree model
    model = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        criterion='gini',
        random_state=42
    )

    # 3. Perform Cross-Validation on the training data
    # We use the SMOTE-Tomek data (X_train_opt, y_train_opt)
    try:
        # Use 3-fold CV for reasonable speed
        scores = cross_val_score(model, X_train_opt, y_train_opt, cv=3, scoring='accuracy', error_score='raise')
        return scores.mean()
    except Exception as e:
        print(f"Error during CV: {e}")
        return 0.0

# --- Load and Prepare Data for Optimization ---

# Select the SMOTE-Tomek dataset for optimization
X_train_opt = data['X_train_smote_tomek']
y_train_opt = data['y_train_smote_tomek']

# 1. Define the search space (Parameter Bounds)
pbounds = {
    # Max Depth (controls tree complexity/overfitting)
    'max_depth': (5, 25),
    
    # Minimum samples required to split an internal node
    'min_samples_split': (5, 50),
    
    # Minimum samples required to be at a leaf node (a strict form of pruning)
    'min_samples_leaf': (1, 20),
}

print(f"Starting Bayesian Optimization for Decision Tree on SMOTE-Tomek dataset...")
print(f"Search space: {pbounds}")
print("-" * 50)

# 2. Initialize the Bayesian Optimizer
optimizer = BayesianOptimization(
    f=dt_cv_score,        # The function to maximize
    pbounds=pbounds,      # The parameter space
    random_state=42,      
    verbose=2             
)

# 3. Run the Optimization
start_time = time.time()
optimizer.maximize(
    init_points=5, # Initial random points
    n_iter=15      # Optimization steps
)
end_time = time.time()

print("\n" + "="*50)
print(f"BAYESIAN OPTIMIZATION COMPLETE in {end_time - start_time:.2f} seconds.")
print("="*50)

# --- Extract Best Parameters and Final Evaluation ---

best_params_raw = optimizer.max['params']
best_score = optimizer.max['target']

# Clean up and finalize the best parameters
best_params = {
    'max_depth': int(round(best_params_raw['max_depth'])),
    'min_samples_split': int(round(best_params_raw['min_samples_split'])),
    'min_samples_leaf': int(round(best_params_raw['min_samples_leaf'])),
    'criterion': 'gini',
    'random_state': 42
}

print(f"Optimal CV Accuracy Found: **{best_score:.4f}**")
print("Optimal Hyperparameters:")
for key, value in best_params.items():
    print(f"  - {key}: {value}")
    
# --- Final Model Training with Optimal Parameters ---

print("\n" + "-"*50)
print("FINAL DECISION TREE TRAINING on SMOTE-TOMEK with Optimized Parameters")
print("-" * 50)

# 1. Train the final model
final_dt_model = DecisionTreeClassifier(**best_params)
final_dt_model.fit(data['X_train_smote_tomek'], data['y_train_smote_tomek'])

# 2. Evaluate on Validation Set
X_val = data['X_val']
y_val = data['y_val']

y_val_pred = final_dt_model.predict(X_val)
final_val_acc = accuracy_score(y_val, y_val_pred)

print(f"✅ Final Optimized Validation Accuracy: **{final_val_acc:.4f}**")

Starting Bayesian Optimization for Decision Tree on SMOTE-Tomek dataset...
Search space: {'max_depth': (5, 25), 'min_samples_split': (5, 50), 'min_samples_leaf': (1, 20)}
--------------------------------------------------
|   iter    |  target   | max_depth | min_sa... | min_sa... |
-------------------------------------------------------------
| 1         | 0.9160115 | 12.490802 | 47.782143 | 14.907884 |
| 2         | 0.9379832 | 16.973169 | 12.020838 | 3.9638958 |
| 3         | 0.8654597 | 6.1616722 | 43.977926 | 12.421185 |
| 4         | 0.9205380 | 19.161451 | 5.9263022 | 19.428287 |
| 5         | 0.9363643 | 21.648852 | 14.555259 | 4.4546743 |
| 6         | 0.9344810 | 21.962323 | 6.9654517 | 6.8036826 |
| 7         | 0.9279720 | 15.764567 | 17.108101 | 12.168496 |
| 8         | 0.9112866 | 8.6609755 | 5.0       | 6.3674442 |
| 9         | 0.9379832 | 15.039896 | 20.625970 | 1.0       |
| 10        | 0.9364635 | 24.138827 | 26.291459 | 1.0       |
| 11        | 0.9158463 | 24.33477